In [1]:
"""
Unified Interpretability Interface for Causal Language Models

This module provides a consistent interface for multiple interpretability methods
for AutoModelForCausalLM, converting all outputs to SHAP-compatible format.

For causal LMs, we explain: "How does each token contribute to predicting the next token?"
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import List, Optional, Tuple
import warnings


# ============================================================================
# SHAP-Compatible Output Format
# ============================================================================

@dataclass
class UnifiedExplanation:
    """
    Unified format that mimics SHAP output structure.
    Compatible with existing analyze_text() function.
    
    For causal LMs: Explains contribution of each input token to next-token prediction
    """
    values: np.ndarray      # Shape: (1, n_tokens, vocab_size) - attribution scores
    data: List[List[str]]   # Shape: (1, n_tokens) - token strings
    base_values: np.ndarray # Shape: (1, vocab_size) - base prediction values
    output_names: List[str] # Top predicted tokens or all vocab
    
    def __getitem__(self, key):
        """Allow array-like indexing for compatibility"""
        if key == 0:
            return self
        raise IndexError(f"Index {key} out of range")


# ============================================================================
# Helper Functions for Causal LM
# ============================================================================

def _get_next_token_logits(model, input_ids, position=-1):
    """
    Get logits for next token prediction at specified position.
    
    Args:
        position: Position to predict next token for (-1 = last position)
    """
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids)
        # Get logits for next token at specified position
        logits = outputs.logits[0, position, :]  # (vocab_size,)
    return logits


def _get_top_k_tokens(logits, tokenizer, k=10):
    """Get top-k predicted tokens and their probabilities."""
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k=k)
    
    top_tokens = [tokenizer.decode([idx]) for idx in top_indices]
    
    return top_tokens, top_probs.cpu().numpy(), top_indices.cpu().numpy()


# ============================================================================
# Method Implementations for Causal LM
# ============================================================================

def _get_integrated_gradients_causal(text, model, tokenizer, target_token_id=None, 
                                     n_steps=50, position=-1):
    """
    Compute Integrated Gradients for causal LM.
    
    Explains: How does each input token contribute to predicting the next token?
    
    Args:
        text: Input text
        target_token_id: Token ID to explain (None = top predicted token)
        n_steps: Number of integration steps
        position: Position to predict next token for (-1 = last position)
    """
    try:
        from captum.attr import LayerIntegratedGradients
    except ImportError:
        raise ImportError("Captum required. Install: pip install captum")
    
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    input_ids = inputs['input_ids']
    
    # Get next token prediction
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids)
        next_token_logits = outputs.logits[0, position, :]
        
        if target_token_id is None:
            target_token_id = next_token_logits.argmax().item()
    
    # Get embedding layer based on model architecture
    if hasattr(model, 'transformer'):
        # GPT-2 style
        embed_layer = model.transformer.wte
    elif hasattr(model, 'model') and hasattr(model.model, 'embed_tokens'):
        # LLaMA style
        embed_layer = model.model.embed_tokens
    elif hasattr(model, 'model') and hasattr(model.model, 'decoder'):
        # OPT style
        embed_layer = model.model.decoder.embed_tokens
    else:
        raise ValueError("Cannot find embedding layer for this model architecture")
    
    # Wrapper class to make model return just the target logit (scalar)
    class ModelWrapper(torch.nn.Module):
        def __init__(self, model, position, target_token_id):
            super().__init__()
            self.model = model
            self.position = position
            self.target_token_id = target_token_id
        
        def forward(self, input_ids):
            outputs = self.model(input_ids)
            # Return just the scalar logit for the target token
            return outputs.logits[:, self.position, self.target_token_id]
    
    wrapped_model = ModelWrapper(model, position, target_token_id)
    
    # Create Layer IG explainer with the wrapped model
    lig = LayerIntegratedGradients(wrapped_model, embed_layer)
    
    # Compute attributions
    attributions = lig.attribute(
        input_ids,
        n_steps=n_steps,
        internal_batch_size=32
    )
    
    # Sum across embedding dimension to get per-token attribution
    attr_scores = attributions.squeeze().sum(dim=-1).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    
    # Get base value (prediction with zero embeddings)
    with torch.no_grad():
        zero_embeds = torch.zeros((1, input_ids.shape[1], embed_layer.embedding_dim)).to(input_ids.device)
        attention_mask = torch.ones_like(input_ids)
        base_outputs = model(inputs_embeds=zero_embeds, attention_mask=attention_mask)
        base_logits = base_outputs.logits[0, position, :]
        base_values = torch.softmax(base_logits, dim=-1).cpu().numpy()
    
    # Get top predicted tokens as output names
    top_tokens, _, top_indices = _get_top_k_tokens(next_token_logits, tokenizer, k=10)
    # Clean up token representation (remove special characters, whitespace markers)
    output_names = []
    for token in top_tokens:
        print("Token...")
        print(token)
        # Remove Ġ (GPT-2 space marker) and other tokenizer artifacts
        clean_token = token.replace('Ġ', ' ').replace('Â', '').strip()
        if not clean_token:
            clean_token = token  # Keep original if cleaning results in empty string
        output_names.append(clean_token)
    
    return attr_scores, tokens, base_values, output_names, next_token_logits.cpu().numpy()


def _get_attention_weights_causal(text, model, tokenizer, layer=-1, position=-1):
    """
    Extract attention weights from causal LM.
    
    For causal models, attention shows which tokens the model attends to
    when predicting the next token.
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
    
    # Get attention from specified layer
    # Shape: (batch, num_heads, seq_len, seq_len)
    attentions = outputs.attentions[layer][0]
    
    # For causal models, look at attention TO the last token (or specified position)
    # This shows which tokens were important for predicting next token
    if position == -1:
        position = attentions.shape[-1] - 1
    
    # Average across heads, get attention pattern for the position
    token_importance = attentions[:, position, :].mean(dim=0).cpu().numpy()
    
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    # Get next token logits
    next_token_logits = outputs.logits[0, position, :]
    probs = torch.softmax(next_token_logits, dim=-1).cpu().numpy()
    
    # Get top predicted tokens
    top_tokens, _, top_indices = _get_top_k_tokens(next_token_logits, tokenizer, k=10)
    output_names = []
    for token in top_tokens:
        clean_token = token.replace('Ġ', ' ').replace('Â', '').strip()
        if not clean_token:
            clean_token = token
        output_names.append(clean_token)
    
    return token_importance, tokens, probs, output_names, next_token_logits.cpu().numpy()


def _get_gradient_x_input_causal(text, model, tokenizer, target_token_id=None, position=-1):
    """
    Compute Gradient × Input for causal LM.
    """
    try:
        from captum.attr import LayerGradientXActivation
    except ImportError:
        raise ImportError("Captum required. Install: pip install captum")
    
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    input_ids = inputs['input_ids']
    
    # Get next token prediction
    model.eval()
    outputs = model(input_ids)
    next_token_logits = outputs.logits[0, position, :]
    
    if target_token_id is None:
        target_token_id = next_token_logits.argmax().item()
    
    # Get embedding layer
    if hasattr(model, 'transformer'):
        embed_layer = model.transformer.wte
    elif hasattr(model, 'model') and hasattr(model.model, 'embed_tokens'):
        embed_layer = model.model.embed_tokens
    elif hasattr(model, 'model') and hasattr(model.model, 'decoder'):
        embed_layer = model.model.decoder.embed_tokens
    else:
        raise ValueError("Cannot find embedding layer")
    
    # Wrapper class to return just the target logit
    class ModelWrapper(torch.nn.Module):
        def __init__(self, model, position, target_token_id):
            super().__init__()
            self.model = model
            self.position = position
            self.target_token_id = target_token_id
        
        def forward(self, input_ids):
            outputs = self.model(input_ids)
            return outputs.logits[:, self.position, self.target_token_id]
    
    wrapped_model = ModelWrapper(model, position, target_token_id)
    
    # Compute Layer Gradient × Activation with wrapped model
    lgxa = LayerGradientXActivation(wrapped_model, embed_layer)
    attributions = lgxa.attribute(input_ids)
    
    # Sum across embedding dimension
    attr_scores = attributions.squeeze().sum(dim=-1).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    
    # Get base values
    with torch.no_grad():
        zero_embeds = torch.zeros((1, input_ids.shape[1], embed_layer.embedding_dim)).to(input_ids.device)
        attention_mask = torch.ones_like(input_ids)
        base_outputs = model(inputs_embeds=zero_embeds, attention_mask=attention_mask)
        base_logits = base_outputs.logits[0, position, :]
        base_values = torch.softmax(base_logits, dim=-1).cpu().numpy()
    
    # Get output names
    top_tokens, _, top_indices = _get_top_k_tokens(next_token_logits, tokenizer, k=10)
    output_names = []
    for token in top_tokens:
        clean_token = token.replace('Ġ', ' ').replace('Â', '').strip()
        if not clean_token:
            clean_token = token
        output_names.append(clean_token)
    
    return attr_scores, tokens, base_values, output_names, next_token_logits.cpu().numpy()


def _get_layer_integrated_gradients_causal(text, model, tokenizer, target_token_id=None, 
                                           n_steps=20, position=-1):
    """
    Compute Layer Integrated Gradients for causal LM.
    """
    try:
        from captum.attr import LayerIntegratedGradients
    except ImportError:
        raise ImportError("Captum required. Install: pip install captum")
    
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    input_ids = inputs['input_ids']
    
    # Get prediction
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids)
        next_token_logits = outputs.logits[0, position, :]
        
        if target_token_id is None:
            target_token_id = next_token_logits.argmax().item()
    
    # Define forward function
    def forward_func(input_ids):
        outputs = model(input_ids)
        return outputs.logits[:, position, target_token_id]
    
    # Get embedding layer
    if hasattr(model, 'transformer'):
        # GPT-2 style
        embed_layer = model.transformer.wte
    elif hasattr(model, 'model'):
        # LLaMA style
        embed_layer = model.model.embed_tokens
    else:
        raise ValueError("Cannot find embedding layer for this model architecture")
    
    # Use embedding layer for attribution
    lig = LayerIntegratedGradients(forward_func, embed_layer)
    
    attributions = lig.attribute(
        input_ids,
        n_steps=n_steps
    )
    
    # Sum across embedding dimension
    attr_scores = attributions.squeeze().sum(dim=-1).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    
    # Get base values
    with torch.no_grad():
        base_outputs = model(torch.zeros_like(input_ids))
        base_logits = base_outputs.logits[0, position, :]
        base_values = torch.softmax(base_logits, dim=-1).cpu().numpy()
    
    # Get output names
    top_tokens, _, top_indices = _get_top_k_tokens(next_token_logits, tokenizer, k=10)
    output_names = []
    for token in top_tokens:
        clean_token = token.replace('Ġ', ' ').replace('Â', '').strip()
        if not clean_token:
            clean_token = token
        output_names.append(clean_token)
    
    return attr_scores, tokens, base_values, output_names, next_token_logits.cpu().numpy()


def _get_shap_values_causal(text, model, tokenizer):
    """
    Original SHAP implementation for causal LM.
    
    Note: SHAP should already handle causal LMs correctly.
    """
    try:
        import shap
    except ImportError:
        raise ImportError("SHAP required. Install: pip install shap")
    
    explainer = shap.Explainer(model, tokenizer)
    shap_values = explainer([text])
    
    return shap_values


# ============================================================================
# Main Interface Function
# ============================================================================

def get_explanation(
    text: str,
    model,
    tokenizer,
    method: str = "integrated_gradients",
    target_token_id: Optional[int] = None,
    position: int = -1,
    **kwargs
) -> UnifiedExplanation:
    """
    Get feature attributions for causal LM using specified method.
    
    For causal LMs: Explains how each input token contributes to next-token prediction.
    
    Args:
        text: Input text to explain
        model: AutoModelForCausalLM
        tokenizer: Corresponding tokenizer
        method: Interpretability method to use. Options:
            - "integrated_gradients" (recommended, 10-100x faster than SHAP)
            - "attention" (fastest, ~1000x faster, approximation only)
            - "gradient_x_input" (very fast, ~100x faster)
            - "layer_integrated_gradients" (fast, ~5-20x faster)
            - "shap" (slowest, gold standard)
        target_token_id: Token ID to explain (None = predicted token)
        position: Position to predict next token for (-1 = last position)
        **kwargs: Additional arguments:
            - n_steps: Number of steps for IG methods (default: 50 for IG, 20 for LIG)
            - layer: Which attention layer to use (default: -1, last layer)
    
    Returns:
        UnifiedExplanation object compatible with SHAP interface
    
    Example:
        >>> # Explain: How did each token contribute to predicting the next word?
        >>> explanation = get_explanation(
        ...     "The cat sat on the",
        ...     model,
        ...     tokenizer,
        ...     method="integrated_gradients"
        ... )
        >>> # Model is predicting next token (probably "mat" or "floor")
        >>> # Attribution shows which input tokens influenced this prediction
    """
    method = method.lower()
    
    # Route to appropriate method
    if method == "integrated_gradients" or method == "ig":
        n_steps = kwargs.get('n_steps', 50)
        attr_scores, tokens, base_values, output_names, logits = _get_integrated_gradients_causal(
            text, model, tokenizer, target_token_id, n_steps, position
        )
        
    elif method == "attention" or method == "attn":
        layer = kwargs.get('layer', -1)
        attr_scores, tokens, base_values, output_names, logits = _get_attention_weights_causal(
            text, model, tokenizer, layer, position
        )
        
    elif method == "gradient_x_input" or method == "gxi":
        attr_scores, tokens, base_values, output_names, logits = _get_gradient_x_input_causal(
            text, model, tokenizer, target_token_id, position
        )
        
    elif method == "layer_integrated_gradients" or method == "lig":
        n_steps = kwargs.get('n_steps', 20)
        attr_scores, tokens, base_values, output_names, logits = _get_layer_integrated_gradients_causal(
            text, model, tokenizer, target_token_id, n_steps, position
        )
        
    elif method == "shap":
        shap_values = _get_shap_values_causal(text, model, tokenizer)
        # SHAP already returns correct format
        return shap_values
        
    else:
        raise ValueError(
            f"Unknown method: {method}. "
            f"Choose from: integrated_gradients, attention, gradient_x_input, "
            f"layer_integrated_gradients, shap"
        )
    
    # Convert to SHAP-compatible format
    n_tokens = len(tokens)
    vocab_size = len(base_values)
    
    # Reshape values to match SHAP format: (1, n_tokens, vocab_size)
    # For efficiency, we only store attributions for the target token
    values = np.zeros((1, n_tokens, vocab_size))
    
    # Assign attribution scores to the target token
    if target_token_id is not None:
        values[0, :, target_token_id] = attr_scores
    else:
        # Assign to predicted token
        predicted_token = np.argmax(logits)
        values[0, :, predicted_token] = attr_scores
    
    # Create unified explanation object
    explanation = UnifiedExplanation(
        values=values,
        data=[[str(token) for token in tokens]],
        base_values=base_values.reshape(1, -1),
        output_names=output_names
    )
    
    return explanation


# ============================================================================
# Modified analyze_text Function (Drop-in Replacement)
# ============================================================================

def analyze_text_unified(text, model, tokenizer, method="integrated_gradients", 
                        position=-1, **kwargs):
    """
    Drop-in replacement for analyze_text() for causal LMs.
    
    Explains: How does each input token contribute to next-token prediction?
    
    Args:
        text: Input text to analyze
        model: AutoModelForCausalLM
        tokenizer: Corresponding tokenizer
        method: Interpretability method
        position: Position to predict next token for (-1 = last position)
        **kwargs: Additional arguments for the method
    
    Returns:
        highlights: List of (start, end, label, color) tuples
        outputs: Top predicted next tokens
    """
    import matplotlib.pyplot as plt
    from matplotlib import colors as mcolors
    
    # Get explanation in unified format
    explanation = get_explanation(text, model, tokenizer, method=method, 
                                 position=position, **kwargs)
    
    # Extract values (same as original code)
    all_values = explanation.values
    base_values = explanation.base_values
    data_values = explanation.data
    outputs = explanation.output_names
    
    # Pick the predicted token's attributions
    # Find which token has non-zero attributions
    token_with_values = None
    for i in range(all_values.shape[2]):
        if np.any(all_values[0, :, i] != 0):
            token_with_values = i
            break
    
    if token_with_values is None:
        # Fallback: use last token
        all_values = all_values[0, :, -1]
    else:
        all_values = all_values[0, :, token_with_values]
    
    # Normalize values
    min_val = np.min(all_values)
    max_val = np.max(all_values)
    
    if max_val - min_val == 0:
        normalized = np.ones_like(all_values) * 0.5
    else:
        normalized = (all_values - min_val) / (max_val - min_val)
    
    # Choose colormap
    cmap = plt.cm.seismic
    
    # Generate highlights
    current_pos = 0
    highlights = []
    
    for i in range(len(data_values[0])):
        word = data_values[0][i]
        value = all_values[i]
        color = mcolors.to_hex(cmap(normalized[i]))
        
        start = text.find(word, current_pos)
        if start != -1:
            end = start + len(word)
            label = f"{value:.2f}"
            highlights.append((start, end, label, color))
            current_pos = end
    
    return highlights, outputs


# ============================================================================
# Convenience Functions
# ============================================================================

def show_next_token_prediction(text, model, tokenizer, top_k=10):
    """
    Show what the model predicts as the next token.
    Useful for understanding what we're explaining.
    """
    inputs = tokenizer(text, return_tensors="pt")
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        next_token_logits = outputs.logits[0, -1, :]
    
    top_tokens, top_probs, top_indices = _get_top_k_tokens(next_token_logits, tokenizer, k=top_k)
    
    print(f"\nInput text: '{text}'")
    print(f"\nTop {top_k} predicted next tokens:")
    print("-" * 50)
    for i, (token, prob, idx) in enumerate(zip(top_tokens, top_probs, top_indices), 1):
        print(f"{i}. '{token}' (prob: {prob:.4f}, id: {idx})")
    print()


# ============================================================================
# Example Usage
# ============================================================================

if __name__ == "__main__":
    #list_methods()
    
    print("started")

    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = "gpt2"  # small test model
    model = AutoModelForCausalLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    text = "The cat sat on the"

    # See what's being predicted
    show_next_token_prediction(text, model, tokenizer)

#     # Get explanation using Integrated Gradients (recommended)
#     explanation = get_explanation(
#         text, 
#         model, 
#         tokenizer, 
#         method="integrated_gradients"
#     )

#     # Or use the unified interface
#     highlights, outputs = analyze_text_unified(
#         text,
#         model,
#         tokenizer,
#         method="integrated_gradients"
#     )

    # Benchmark all methods
    #results = benchmark_methods(text, model, tokenizer)


started


/n/home10/jnair/.conda/envs/env1206/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Input text: 'The cat sat on the'

Top 10 predicted next tokens:
--------------------------------------------------
1. ' floor' (prob: 0.0764, id: 4314)
2. ' bed' (prob: 0.0653, id: 3996)
3. ' couch' (prob: 0.0541, id: 18507)
4. ' ground' (prob: 0.0521, id: 2323)
5. ' edge' (prob: 0.0478, id: 5743)
6. ' bench' (prob: 0.0316, id: 7624)
7. ' table' (prob: 0.0315, id: 3084)
8. ' sofa' (prob: 0.0288, id: 34902)
9. ' other' (prob: 0.0199, id: 584)
10. ' back' (prob: 0.0190, id: 736)



In [2]:

sample_corpus = """
Jane Smith
Concurrent AB/SM in Computer Science, secondary in Government. GPA: 3.XX	           May 2026
Relevant Coursework: Optimal Transport for Machine Learning, Advanced Computer Vision, Interpretable AI, Machine Learning, Computing At Scale, Algorithms at Ends of the Wire, Applied Privacy for Data Science


Technical Skills

Languages: Python, SQL, Java, HTML/CSS/JavaScript, bash scripting, OCaml, C, MATLAB
Libraries: PyTorch, Keras, pandas, scikit-learn, Matplotlib, OpenCV, NumPy
Certifications: Introduction to Data Science in Python; Applied Plotting, Charting & Data Representation in Python

Experience                                                                                                                                                                    Washington, D.C.
Consulting Intern                                                                                                                                                         Jun 2025 – Aug 2025
•	Analyzed data and created client-facing materials in support of operational and strategic initiatives.

Data Analytics Group                                                                                                            Nowhere, MA
Chief Operations Officer	          Sep 2023 – Jun 2025
•	COO of data consulting organization; managed recruitment and promotion processes for over 75 members; oversaw all internal departments.
•	Performed data analysis, visualization, and machine learning work for corporate and nonprofit clients; planned formal for 120 guests.

U.S. Federal Agency                                                                                                                                         Washington, D.C. (Remote)
Program Data Analyst Fellow                                                                                                                           Jun – Aug 2024
•	Developed a Python-based data infrastructure to enhance understanding of long-term care residents and facilities; conducted comprehensive policy analysis, resulting in a 21-page report; selected for CIF’s public Demo Day.


Q: Should Jane Smith be advanced to the next round for the position of Data Analyst?

"""

#analyze_text(sample_corpus)


In [ ]:
### Take highlights and outputs and connect to interface code
import gradio as gr
import re
from datetime import datetime

def highlight_text(text, highlights, outputs, title=""):
    """
    Create HTML with highlighted sections.
    """
    if not highlights:
        content = f'<div style="font-family: Arial, sans-serif; line-height: 1.8; padding: 20px;">{text}</div>'
    else:
        # Sort highlights by start position
        sorted_highlights = sorted(highlights, key=lambda x: x[0])

        result = []
        last_idx = 0

        for start, end, label, color in sorted_highlights:
            # Add unhighlighted text before this highlight
            if last_idx < start:
                result.append(text[last_idx:start].replace('\n', '<br>'))

            # Add highlighted text
            highlighted_span = f'<mark style="background-color: {color}; padding: 2px 4px; border-radius: 3px;" title="{label}">{text[start:end]}</mark>'
            result.append(highlighted_span)
            last_idx = end

        # Add remaining unhighlighted text
        if last_idx < len(text):
            result.append(text[last_idx:].replace('\n', '<br>'))

        content = ''.join(result)

        # Get unique labels for legend
        unique_labels = {}
        for _, _, label, color in sorted_highlights:
            if label not in unique_labels:
                unique_labels[label] = color

        legend = f"""
        <div style="margin-top: 20px; padding: 10px; background-color: #f5f5f5; border-radius: 5px;">
            <strong>Legend:</strong><br>
            {' '.join([f'<mark style="background-color:#000000; padding: 2px 8px; margin: 5px; border-radius: 3px;">{output}</mark>' for output in outputs])}
        </div>
        """

    # Add title if provided
    title_html = f'<h3 style="color: #555; margin-bottom: 10px;">{title}</h3>' if title else ''

    # Create HTML with styling
    html_content = f"""
    {title_html}
    <div style="font-family: Arial, sans-serif; line-height: 1.8; padding: 20px; border: 1px solid #ddd; border-radius: 5px; background-color: white;">
        {content}
    </div>
    {legend if highlights else ''}
    """

    return html_content


def process_text(text):
    """
    Main function that processes text and returns highlighted HTML.
    This is called whenever the text is edited.
    """
    if not text.strip():
        return "<p>Enter some text to analyze...</p>"

    # Run your analysis function
    highlights, outputs = analyze_text_unified(text, model, tokenizer, method="integrated_gradients")

    # Generate highlighted HTML
    return highlight_text(text, highlights, outputs)


# Sample resume text
sample_text = sample_corpus

# Store original text and saved versions
original_text = sample_text
saved_versions = []

def reset_text():
    """Reset to original text."""
    return original_text, process_text(original_text)

def save_current_version(text, html):
    """Save the current text and ALREADY-RENDERED analysis."""
    global saved_versions
    if not text.strip():
        return gr.update(), "⚠️ Cannot save empty text"

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Use the pre-rendered HTML instead of re-analyzing
    html_with_title = f'<h3 style="color: #555; margin-bottom: 10px;">Saved: {timestamp}</h3>{html}'

    saved_versions.append({
        'text': text,
        'html': html_with_title,
        'timestamp': timestamp
    })

    choices = [f"Version {i+1}: {v['timestamp']}" for i, v in enumerate(saved_versions)]

    return gr.update(choices=choices, value=None), f"✅ Saved version {len(saved_versions)}"

def load_saved_version(selected):
    """Load a saved version for comparison."""
    if not selected or not saved_versions:
        return "<p>No saved version selected</p>"

    # Extract version number from selection
    version_num = int(selected.split(":")[0].replace("Version ", "")) - 1
    return saved_versions[version_num]['html']

def clear_comparison():
    """Clear the comparison view."""
    return "<p>No comparison loaded. Save a version and select it from the dropdown to compare.</p>"

# Create Gradio interface with Blocks for more control
with gr.Blocks(title="Resume Screener Auditor Tool") as demo:
    gr.Markdown("# Resume Screener Auditor Tool")
    gr.Markdown("Test out this interpretable resume screener algorithm on different inputs, save, and compare the results.")

    with gr.Row():
        # Left column - Text editor
        with gr.Column(scale=1):
            gr.Markdown("### Text Editor")
            text_input = gr.Textbox(
                value=sample_text,
                lines=20,
                label="Edit Resume to Assess",
                placeholder="Paste or type your text here..."
            )
            with gr.Row():
                reset_btn = gr.Button("🔄 Revert to Original", variant="secondary")
                save_btn = gr.Button("💾 Save Current Version", variant="primary")
            save_status = gr.Markdown("")

        # Middle column - Current analysis (updates in real-time)
        with gr.Column(scale=1):
            gr.Markdown("### Current Analysis (Live)")
            html_output = gr.HTML(label="Current Highlights", value=process_text(sample_text))

        # Right column - Saved version for comparison
        with gr.Column(scale=1):
            gr.Markdown("### Saved Version (Comparison)")
            version_dropdown = gr.Dropdown(
                choices=[],
                label="Select saved version to compare",
                interactive=True
            )
            clear_btn = gr.Button("🗑️ Clear Comparison", variant="secondary")
            comparison_output = gr.HTML(
                label="Saved Highlights",
                value="<p>No comparison loaded. Save a version and select it from the dropdown to compare.</p>"
            )

    # Update current highlights when text changes
    text_input.change(
        fn=process_text,
        inputs=text_input,
        outputs=html_output
    )

    # Reset button functionality
    reset_btn.click(
        fn=reset_text,
        inputs=None,
        outputs=[text_input, html_output]
    )

    # Save button functionality
    save_btn.click(
        fn=save_current_version,
        inputs=[text_input, html_output],  # Pass both text AND html
        outputs=[version_dropdown, save_status]
    )

    # Load saved version for comparison
    version_dropdown.change(
        fn=load_saved_version,
        inputs=version_dropdown,
        outputs=comparison_output
    )

    # Clear comparison button
    clear_btn.click(
        fn=clear_comparison,
        inputs=None,
        outputs=comparison_output
    )


# Launch the interface
demo.launch(debug=True, share=True)

Token...
 the
Token...
 a
Token...
 data
Token...
 machine
Token...
 this
Token...
 Python
Token...
 computational
Token...
 Machine
Token...
 an
Token...
 new
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://e73477295b7651cd8f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
